# 🚀 Fine-Tuning ModernBERT-Base (164M) Multi-Task Risk Taxonomy for Code Oracle — Multi-Task Base
### Real-World Commit Revert/Hotfix + Hard Negative Training, Early Stopping, & Held-Out Benchmark

This notebook executes real fine-tuning of `answerdotai/ModernBERT-base` (164M parameters) using a **Multi-Task Architecture** (Continuous Risk Regression, 5-Class Risk Taxonomy, and Epistemic Uncertainty Estimation).

#### 🌟 Key Training Characteristics:
1. **Hard Negative Filtering (Stage 1-2 Symbolic Gate):** 100% of negative samples pass AST and cycle checks. The model only learns to resolve subtle semantic risks.
2. **Real-World Commits & Surviving Mutants:** Mined and harvested across Python, TypeScript, Go, and Rust.
3. **Multi-Task Risk Taxonomy (ADR-0003):** 5 hazard classes (`BreakingPublicAPI`, `SecuritySurface`, `ConcurrencyHazard`, `PerformanceRegression`, `SilentLogicDrift`).
4. **Class Loss Re-weighting (`pos_weight = 2.0`):** Heavily penalizes false negatives (missed subtle bugs) to maximize recall on critical security and logic bugs.
5. **Extended 8-Epoch Training & Checkpoint Tracking:** Early stopping (patience=3) and validation loss checkpoint tracking.
6. **Post-Hoc Temperature Scaling:** Calibrated epistemic confidence scores smoothly clamped in `[0.8, 2.5]` via L-BFGS.
7. **Independent Held-Out Benchmark & PR Sweep:** Evaluated on 400 unseen samples from independent repos (Flask, Httpx, Fastify, Chi, Serde) with full Precision-Recall sweep across thresholds `[0.25 .. 0.60]` and default threshold 0.40.
- **Target Hardware:** Free Google Colab T4 GPU (~3-7 minutes total training time).


## 1. Verify Free Google Colab T4 GPU
Ensure runtime is configured to use GPU (`Runtime > Change runtime type > T4 GPU`).


In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Device:   {device}")


## 2. Install Required Packages


In [ ]:
!pip install -q -U "transformers>=4.48.0" datasets safetensors accelerate scikit-learn
import transformers
print(f"[✓] Transformers loaded: v{transformers.__version__}")


## 3. Unpack Code Oracle Dataset (Train, Val, & Held-Out Benchmark)


In [ ]:
import os, base64, io, zipfile, json
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

#@title 📦 Dataset Variant Selection
# Choose between Golden Hybrid (v3-hybrid: ~4,500 samples), Medium Scale (v3-medium: 5,000 samples), or Full Scale (v3-full: 10,000 samples)
DATASET_VARIANT = "v3_hybrid" #@param ["v3_hybrid", "v3_medium", "v3_full"]

EMBEDDED_ZIP_B64_HYBRID = "..."
EMBEDDED_ZIP_B64_MEDIUM = "..."
EMBEDDED_ZIP_B64_FULL = "..."

if DATASET_VARIANT == "v3_full":
    chosen_b64 = EMBEDDED_ZIP_B64_FULL
elif DATASET_VARIANT == "v3_medium":
    chosen_b64 = EMBEDDED_ZIP_B64_MEDIUM
else:
    chosen_b64 = EMBEDDED_ZIP_B64_HYBRID
zip_bytes = base64.b64decode(chosen_b64)
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    zf.extractall(DATA_DIR)

train_path = DATA_DIR / "dataset_train.jsonl"
val_path = DATA_DIR / "dataset_val.jsonl"
heldout_path = DATA_DIR / "dataset_heldout_eval.jsonl"

with open(train_path, "r", encoding="utf-8") as f:
    train_records = [json.loads(line) for line in f if line.strip()]

with open(val_path, "r", encoding="utf-8") as f:
    val_records = [json.loads(line) for line in f if line.strip()]

with open(heldout_path, "r", encoding="utf-8") as f:
    heldout_records = [json.loads(line) for line in f if line.strip()]

print(f"[✓] Successfully unpacked Code Oracle {DATASET_VARIANT} dataset:")
print(f"    - Training Set:   {len(train_records):>5} samples (100% passed Stage 1-2 symbolic gate)")
print(f"    - Validation Set: {len(val_records):>5} samples (50% PASS / 50% REJECT)")
print(f"    - Held-Out Eval:  {len(heldout_records):>5} samples (Independent unseen repos)")


## 4. Define PyTorch Multi-Task Model & Dataset


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

MODEL_ID = "answerdotai/ModernBERT-base"
TAXONOMY_CLASSES = [
    "BreakingPublicAPI",
    "SecuritySurface",
    "ConcurrencyHazard",
    "PerformanceRegression",
    "SilentLogicDrift",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

class CodeOracleDataset(Dataset):
    def __init__(self, records, max_length=512):
        self.records = records
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        dsl = rec.get('input_dsl', '')
        enc = tokenizer(
            dsl,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['label'] = torch.tensor(rec.get('label', 1), dtype=torch.long)
        item['risk_target'] = torch.tensor([rec.get('risk_score', 0.1)], dtype=torch.float32)

        tax_labels = rec.get('taxonomy_labels', {})
        tax_vec = [float(tax_labels.get(c, 0.0)) for c in TAXONOMY_CLASSES]
        item['taxonomy_target'] = torch.tensor(tax_vec, dtype=torch.float32)
        return item

class ModernBERTMultiTaskModel(nn.Module):
    def __init__(self, encoder_name=MODEL_ID):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden_size = self.encoder.config.hidden_size  # 768

        # Head 1: Continuous Risk Regression (0.0 to 1.0)
        self.risk_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Linear(256, 1),
        )

        # Head 2: Multi-Label Risk Taxonomy (5 classes)
        self.taxonomy_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Linear(256, len(TAXONOMY_CLASSES)),
        )

        # Head 3: Epistemic Uncertainty (Heteroscedastic log-variance)
        self.uncertainty_head = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Linear(128, 1),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        h_pool = sum_embeddings / sum_mask

        risk_raw = self.risk_head(h_pool)
        risk_score = torch.sigmoid(risk_raw)

        taxonomy_logits = self.taxonomy_head(h_pool)
        taxonomy_probs = torch.sigmoid(taxonomy_logits)

        s = self.uncertainty_head(h_pool)
        log_variance = torch.clamp(s, min=-6.0, max=6.0)
        variance = torch.exp(log_variance)
        confidence = 1.0 - torch.clamp(torch.sqrt(variance), min=0.0, max=1.0)

        return {
            'risk_score': risk_score,
            'risk_logits': risk_raw,
            'taxonomy_logits': taxonomy_logits,
            'taxonomy_probs': taxonomy_probs,
            'log_variance': log_variance,
            'confidence': confidence,
        }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ModernBERTMultiTaskModel(MODEL_ID).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"[✓] ModernBERT Multi-Task Model initialized on {device}!")
print(f"    Total Parameters: {total_params:,} (~{total_params * 2 / (1024**2):.1f} MB in BF16)")


## 5. Execute Real PyTorch Fine-Tuning (pos_weight = 2.0, Early Stopping, & Checkpoint Tracking)
Trains for up to 8 epochs with cosine learning rate schedule, heteroscedastic multi-task loss with **`pos_weight = 2.0`**, validation tracking after each epoch, and early stopping (patience=3).


In [ ]:
import copy
import time
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

batch_size = 16
epochs = 8
lr = 3e-5
patience = 3

if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Defensive guard: auto-reload records if kernel was restarted
if 'train_records' not in globals() or 'val_records' not in globals():
    import json
    data_dir = Path("/content/data")
    if not (data_dir / "dataset_train.jsonl").exists() and "EMBEDDED_ZIP_B64" in globals():
        import base64, io, zipfile
        data_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(base64.b64decode(EMBEDDED_ZIP_B64))) as zf:
            zf.extractall(data_dir)
    with open(data_dir / "dataset_train.jsonl", "r", encoding="utf-8") as f:
        train_records = [json.loads(line) for line in f if line.strip()]
    with open(data_dir / "dataset_val.jsonl", "r", encoding="utf-8") as f:
        val_records = [json.loads(line) for line in f if line.strip()]

if 'model' not in globals():
    model = ModernBERTMultiTaskModel(MODEL_ID).to(device)

train_ds = CodeOracleDataset(train_records)
val_ds = CodeOracleDataset(val_records)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

use_amp = torch.cuda.is_available()
amp_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and amp_dtype == torch.float16))

best_val_loss = float('inf')
best_val_acc = 0.0
best_model_state = None
patience_counter = 0
# pos_weight = 2.0 (ADR-0003: heavily penalizes missed subtle bugs)
pos_weight_tax = torch.ones(len(TAXONOMY_CLASSES), device=device) * 2.0

print(f"[*] Starting Real PyTorch Fine-Tuning across up to {epochs} Epochs...")
print(f"    Batch Size: {batch_size} | Max Steps: {total_steps} | Patience: {patience} | pos_weight: 2.0 | Device: {device}")

t0_start = time.perf_counter()

for ep in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    ep_start = time.perf_counter()

    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        risk_target = batch['risk_target'].to(device)
        taxonomy_target = batch['taxonomy_target'].to(device)

        with torch.cuda.amp.autocast(enabled=use_amp, dtype=amp_dtype):
            out = model(input_ids, attention_mask)
            # Heteroscedastic multi-task loss with pos_weight = 2.0
            risk_pred = out['risk_score'].view(-1, 1)
            risk_t = risk_target.view(-1, 1).float()
            s = out['log_variance'].view(-1, 1)
            tax_t = taxonomy_target.float()

            l_tax = F.binary_cross_entropy_with_logits(out['taxonomy_logits'], tax_t, pos_weight=pos_weight_tax)
            risk_weight = torch.where(risk_t >= 0.5, 2.0, 1.0)
            l_risk = torch.mean(risk_weight * F.huber_loss(risk_pred, risk_t, delta=0.1, reduction='none'))
            diff_sq = (risk_t - risk_pred) ** 2
            l_unc = torch.mean(0.5 * torch.exp(-s) * diff_sq + 0.5 * s)
            loss = l_risk + l_tax + l_unc

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation Loop
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            risk_target = batch['risk_target'].to(device)
            taxonomy_target = batch['taxonomy_target'].to(device)
            labels = batch['label'].to(device)

            with torch.cuda.amp.autocast(enabled=use_amp, dtype=amp_dtype):
                out = model(input_ids, attention_mask)
                risk_pred = out['risk_score'].view(-1, 1)
                risk_t = risk_target.view(-1, 1).float()
                s = out['log_variance'].view(-1, 1)
                tax_t = taxonomy_target.float()

                l_tax = F.binary_cross_entropy_with_logits(out['taxonomy_logits'], tax_t, pos_weight=pos_weight_tax)
                risk_weight = torch.where(risk_t >= 0.5, 2.0, 1.0)
                l_risk = torch.mean(risk_weight * F.huber_loss(risk_pred, risk_t, delta=0.1, reduction='none'))
                diff_sq = (risk_t - risk_pred) ** 2
                l_unc = torch.mean(0.5 * torch.exp(-s) * diff_sq + 0.5 * s)
                v_loss = l_risk + l_tax + l_unc
                val_loss += v_loss.item()

                pred_choice = (out['risk_score'] < 0.5).long().squeeze(-1)
                correct += (pred_choice == labels).sum().item()
                total += labels.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total * 100.0
    ep_dur = time.perf_counter() - ep_start
    print(f"🔥 Epoch {ep}/{epochs} ({ep_dur:.1f}s) | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.1f}%")

    # Checkpoint tracking & Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"    ✨ Val loss improved to {avg_val_loss:.4f} (Acc: {val_acc:.1f}%). Checkpoint saved.")
    else:
        patience_counter += 1
        print(f"    ⚠️ Val loss did not improve ({patience_counter}/{patience}).")
        if patience_counter >= patience:
            print(f"    🛑 Early stopping triggered at epoch {ep}! Restoring best checkpoint.")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"[✓] Restored best model checkpoint (Val Loss: {best_val_loss:.4f}, Val Acc: {best_val_acc:.1f}%)")

t_total = time.perf_counter() - t0_start
print(f"\n🏆 Training Complete in {t_total:.1f} seconds!")


## 6. Post-Hoc Temperature Scaling Calibration
Fits a temperature parameter $T$ on validation logits via L-BFGS to calibrate the epistemic confidence score. Smooth parametrization strictly clamps $T \in [0.8, 2.5]$.


In [ ]:
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if 'use_amp' not in globals():
    use_amp = torch.cuda.is_available()
    amp_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
if 'val_ds' not in globals():
    if 'val_records' not in globals():
        import json
        data_dir = Path("/content/data")
        with open(data_dir / "dataset_val.jsonl", "r", encoding="utf-8") as f:
            val_records = [json.loads(line) for line in f if line.strip()]
    val_ds = CodeOracleDataset(val_records)

val_loader_eval = DataLoader(val_ds, batch_size=16, shuffle=False)
val_logits = []
val_targets = []

model.eval()
with torch.no_grad():
    for batch in val_loader_eval:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        with torch.cuda.amp.autocast(enabled=use_amp, dtype=amp_dtype):
            out = model(input_ids, attention_mask)
        val_logits.append(out['risk_logits'])
        # Target: 1.0 if actual bug (label=0), 0.0 if clean (label=1)
        val_targets.append((1.0 - batch['label'].float().to(device)).unsqueeze(-1))

val_logits = torch.cat(val_logits, dim=0)
val_targets = torch.cat(val_targets, dim=0)

# Optimize temperature T via L-BFGS with smooth clamp strictly bounded in [0.8, 2.5]
raw_temp = nn.Parameter(torch.zeros(1, device=device))
optimizer_t = torch.optim.LBFGS([raw_temp], lr=0.05, max_iter=50)
nll_criterion = nn.BCEWithLogitsLoss()

def eval_t():
    optimizer_t.zero_grad()
    # Smooth parametrization guarantees T in (0.8, 2.5)
    t_bounded = 0.8 + 1.7 * torch.sigmoid(raw_temp)
    loss = nll_criterion(val_logits / t_bounded, val_targets)
    loss.backward()
    return loss

optimizer_t.step(eval_t)
calibrated_T = float((0.8 + 1.7 * torch.sigmoid(raw_temp)).item())

print(f"[✓] Post-hoc Temperature Scaling calibrated on validation set:")
print(f"    Optimal Temperature T = {calibrated_T:.4f} (bounded in [0.8, 2.5])")


## 7. Independent Held-Out Benchmark & Precision-Recall Sweep
Evaluates model performance against **400 unseen real-world commits & mutations** from Flask, Httpx, Fastify, Chi, and Serde.
Computes a full Precision-Recall sweep across thresholds `[0.25 .. 0.60]` and reports metrics at configurable default threshold (0.40).


In [ ]:
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if 'use_amp' not in globals():
    use_amp = torch.cuda.is_available()
    amp_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
if 'calibrated_T' not in globals():
    calibrated_T = 1.0  # Fallback uncalibrated temperature
if 'heldout_records' not in globals():
    import json
    data_dir = Path("/content/data")
    if not (data_dir / "dataset_heldout_eval.jsonl").exists() and "EMBEDDED_ZIP_B64" in globals():
        import base64, io, zipfile
        data_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(base64.b64decode(EMBEDDED_ZIP_B64))) as zf:
            zf.extractall(data_dir)
    with open(data_dir / "dataset_heldout_eval.jsonl", "r", encoding="utf-8") as f:
        heldout_records = [json.loads(line) for line in f if line.strip()]

heldout_ds = CodeOracleDataset(heldout_records)
heldout_loader = DataLoader(heldout_ds, batch_size=16, shuffle=False)

model.eval()
y_true = []
all_risks = []
all_confs = []

with torch.no_grad():
    for batch in heldout_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        with torch.cuda.amp.autocast(enabled=use_amp, dtype=amp_dtype):
            out = model(input_ids, attention_mask)

        y_true.extend(labels.cpu().tolist())
        # Apply calibrated temperature scaling to match runtime decision engine
        scaled_r = torch.sigmoid(out['risk_logits'] / calibrated_T)
        all_risks.extend(scaled_r.cpu().squeeze(-1).tolist())
        all_confs.extend(out['confidence'].cpu().squeeze(-1).tolist())

# Configurable Decision Threshold (Default: 0.40 for higher bug catch rate)
DEFAULT_THRESHOLD = 0.40

def evaluate_at_threshold(threshold):
    y_pred = [1 if r < threshold else 0 for r in all_risks]
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)

    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        'threshold': threshold,
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'specificity': spec,
        'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn,
    }

THRESHOLDS = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
sweep_results = [evaluate_at_threshold(t) for t in THRESHOLDS]

print("=" * 96)
print(" PRECISION-RECALL THRESHOLD SWEEP (HELD-OUT BENCHMARK)")
print("=" * 96)
print(f"{'Threshold':<14} | {'Accuracy':<10} | {'Precision':<10} | {'Recall (Safe)':<14} | {'Spec (Bugs)':<12} | {'F1':<8} | {'Caught':<7} | {'Missed':<7} | {'False Alarms':<12}")
print("-" * 96)
for row in sweep_results:
    t_label = f"{row['threshold']:.2f}"
    if abs(row['threshold'] - DEFAULT_THRESHOLD) < 1e-4:
        t_label += " ⚡️ (Def)"
    elif abs(row['threshold'] - 0.50) < 1e-4:
        t_label += " ⭐️ (Base)"
    print(f"{t_label:<14} | {row['accuracy']*100:>8.2f}% | {row['precision']*100:>8.2f}% | {row['recall']*100:>12.2f}% | {row['specificity']*100:>10.2f}% | {row['f1']:>8.4f} | {row['TN']:>6} | {row['FP']:>6} | {row['FN']:>12}")
print("=" * 96)

m_def = evaluate_at_threshold(DEFAULT_THRESHOLD)
print(f"\n[★] Selected Operating Point (Threshold = {DEFAULT_THRESHOLD}):")
print(f"    - Accuracy:        {m_def['accuracy'] * 100:.2f}%")
print(f"    - Specificity:     {m_def['specificity'] * 100:.2f}% ({m_def['TN']}/200 bugs caught, {m_def['FP']} missed)")
print(f"    - Precision:       {m_def['precision'] * 100:.2f}%")
print(f"    - Recall (Safe):   {m_def['recall'] * 100:.2f}% ({m_def['TP']}/200 safe approved, {m_def['FN']} false alarms)")


## 8. Package Weights & Download Package for Code Oracle


In [ ]:
import shutil
from safetensors.torch import save_file
from google.colab import files

if 'calibrated_T' not in globals():
    calibrated_T = 1.0
if 'DEFAULT_THRESHOLD' not in globals():
    DEFAULT_THRESHOLD = 0.40
if 'm_def' not in globals():
    m_def = {'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'specificity': 0.0, 'TP': 0, 'FP': 0, 'TN': 0, 'FN': 0}
if 'sweep_results' not in globals():
    sweep_results = []

EXPORT_DIR = Path("/content/weights_multitask_base")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save safetensors weights
weights_file = EXPORT_DIR / "model.safetensors"
save_file(model.state_dict(), str(weights_file))

# 2. Save tokenizer
tokenizer.save_pretrained(str(EXPORT_DIR))

# 3. Save config with calibration & threshold settings
config_data = {
    "encoder": MODEL_ID,
    "hidden_size": 768,
    "num_taxonomy_classes": len(TAXONOMY_CLASSES),
    "taxonomy_classes": TAXONOMY_CLASSES,
    "model_name": "code-oracle-laya-modernbert-base-v3",
    "calibrated_temperature": round(calibrated_T, 4),
    "default_decision_threshold": DEFAULT_THRESHOLD,
    "evaluation_metrics": {
        "threshold": DEFAULT_THRESHOLD,
        "accuracy": round(m_def["accuracy"], 4),
        "precision": round(m_def["precision"], 4),
        "recall": round(m_def["recall"], 4),
        "f1": round(m_def["f1"], 4),
        "specificity": round(m_def["specificity"], 4),
        "confusion_matrix": {"TP": m_def["TP"], "FP": m_def["FP"], "TN": m_def["TN"], "FN": m_def["FN"]},
    },
    "threshold_sweep": sweep_results,
}
with open(EXPORT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config_data, f, indent=2)

# 4. Zip and trigger download
zip_out = "/content/code_oracle_laya_multitask_weights.zip"
shutil.make_archive('/content/code_oracle_laya_multitask_weights', 'zip', EXPORT_DIR)

print(f"[✓] Weights successfully exported to {EXPORT_DIR}!")
print(f"[✓] Archive created: {zip_out} (~{Path(zip_out).stat().st_size / (1024**2):.1f} MB)")
print('[*] Triggering automatic download...')
files.download(zip_out)
